Install libraries from requirements.txt

In [1]:
pip install -r requirements.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from snowflake.snowpark import Session
import json
import os
import re
import io
import uuid

import pandas as pd
from openpyxl import load_workbook

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
import pypdf
from PIL import Image, ImageFilter, ImageEnhance
from pdf2image import convert_from_path


In [3]:
BASE_DIR = os.getcwd()

excel_path      = os.path.join(BASE_DIR, "invoice_extract.xlsx")
input_directory = r'C:\\Users\\jakub.klosowski\\OneDrive - Autoliv\\Desktop\\Invoices\\Courier Network'
output_directory = os.path.join(BASE_DIR, "output")

# Tesseract executable path (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path for PDF conversion (Windows)
POPPLER_PATH = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

print("Excel Path:      ", excel_path)
print("Input Directory: ", input_directory)


Excel Path:       c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\invoice_extract.xlsx
Input Directory:  C:\\Users\\jakub.klosowski\\OneDrive - Autoliv\\Desktop\\Invoices\\Courier Network


In [4]:
try:
    version = pytesseract.get_tesseract_version()
    print(f"Tesseract version: {version}")
except Exception as e:
    print(f"Tesseract not found or misconfigured: {e}")


Tesseract version: 5.5.0.20241111


Update snowflake credentials

In [6]:
connection_parameters = {
    "user":          "JAKUB.KLOSOWSKI@AUTOLIV.COM",
    "authenticator": "externalbrowser",
    "account":       "VN02639-YR60386",
    "warehouse":     "DEV_PRINCIPAL_WH",
    "database":      "PROTO_DEVTEAM_DB",
    "schema":        "PUBLIC",
}

session = Session.builder.configs(connection_parameters).create()
print("Snowflake session created.")


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/f3603cca-3b48-4625-afd2-2eb0587bf1d6/saml2?SAMLRequest=nZLBctowEIZfxaOebcs2oaDBZAyUlmnSMuC0096EtQYNsuRKcgw8fYUdZtJDcujNI3%2B7%2BrT%2FTu5PlfCeQRuuZIqiACMPZKEYl%2FsUPeVLf4Q8Y6lkVCgJKTqDQffTiaGVqEnW2IPcwJ8GjPVcI2lI9yNFjZZEUcMNkbQCQ2xBttnjA4kDTGqtrCqUQK9K3q%2BgxoC2zvBWwgx3egdraxKGbdsGbRIovQ9jjHGIx6GjrsiHG39yb3qDj0I8uPKOcPj6xW3GZT%2BC97R2PWTIlzxf%2B%2Bvv2xx52U11rqRpKtBb0M%2B8gKfNQy9gnMGvzRAno2HQurn50GhVQ0AvjYbASNWWgh6hUFXdWNc9cF9hCSwUas%2FdAFaLFNVHzvDskm9OCT3i04hVWWu3c%2Fx5%2BfXn%2BXKuFW1W2V59mtHDZbAsCuT9uCUcXxNeGdPASl5zte4Ix0MfJ36MczwmSUKiKEjGyW%2FkLZwfl9R2lTf5ziOoeKGVUaVVUnAJvWXinlUU1E92g5E%2FGMZ3Pi1Z7Meww3ejj7syYsPwmnaM%2Bg0inYie%2Fu9cJuHrLi9L%2Bc3ltFqsleDF2VsqXVH7doxREHUnnPllhxKoKBcZYxqMcXEKodq5Bmrd7lvdAAqn%2Fa3%2Fbv%2F0Lw%3D%3D&RelayState=ver%3A3-hint%3A3302906936078-ETMsDgAAAZ0Kl3VhABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEAa7GjYd5cMahEhVon2UleYAAACQ

Test LLM Response

In [7]:
prompt = "What is your age?"

sql = f"""
    SELECT snowflake.cortex.complete(
        'llama3.1-70b',
        $$ {prompt.strip()} $$
    )
"""

# Execute the query and collect the result
result = session.sql(sql).collect()

# Print the LLM-generated response
print(result[0][0])

I was released to the public in 2023.


In [5]:
def ask_cortex_invoice_data(session, extracted_text):
    try:
        prompt = f"""
Return only the JSON.

The following is an OCR extract of a Courier Network Inc. invoice. Your task is to extract the following fields with absolute precision.
Each field must be extracted exactly as it appears in the document, even if there are typos or formatting issues in the OCR.

The OCR text preserves the original horizontal layout of the document. The document has THREE columns:
- LEFT column: shipment details (AWB#, Pickup Date, Depart/Arrive Port, Weight, Pieces, Value, Service, Pod, Signed)
- CENTER column: SHIPPER block and CONSIGNEE block (names and addresses)
- RIGHT column: additional references (PTN, QUSD, REQNA, distances)

Read each line carefully and identify which column each value belongs to based on its horizontal position.

Extract the following fields:

1. supplier
   The company issuing the invoice.
   Always set supplier to "Courier Network Inc."

2. invoice_no
   The invoice (Tax Invoice) number.
   Found in the upper portion of the document, on a line containing "Tax Invoice".
   Extract only the alphanumeric identifier after "Tax Invoice" — ignore any trailing words like "Original".

3. invoice_date
   The invoice date.
   Found in the upper-right portion of the document, on a line containing "Invoice Date:".
   Extract only the date value after "Invoice Date:" — e.g. "11/30/25".
   Important: dates are written in the MM/DD/YYYY format (month/day/year) only for invoice_date. However, in the JSON response, always return dates in the DD/MM/YYYY format (day/month/year). For example, if the document shows "09/03/2025", return "03/09/2025" in the JSON.

4. bill_to
   The "Bill To" recipient name and address.
   Always set bill_to to 'Autoliv ASP, Inc. 3350 Airport Road Ogden, UT 84405 USA'. Do not extract from the document.

5. awb
   The Air Waybill number.
   Found in the LEFT column, on a line containing "AWB #:" or "IAWB#:".

7. ship_date
   The pickup date.
   Found in the LEFT column, on a line containing "Pickup Date:".
   Extract only the date value after "Pickup Date:".

8. delivery_date
   The delivery date.
   Found in the LEFT column, on a line containing "Pod:".
   Extract only the date value after "Pod:".

9. delivery_time
    The delivery time.
    Found on the same line as the delivery date (Pod:), typically after the date.
    Extract only the time value — e.g. "14:30:00".

10. delivery_timezone
    The delivery timezone.
    Set N/A for all documents as this is not present in the OCR.
    

11. shipper
   The shipper name and full address.
   The label "SHIPPER:" appears in the CENTER column.
   The shipper content spans the lines BELOW that label in the CENTER column:
   - First line after the label: shipper company name
   - Following lines: street address, city/zip, country/state
   There may be blank lines or gaps between address parts in the OCR — collect ALL lines belonging to the CENTER column in the SHIPPER section until the CONSIGNEE label appears.
   Combine all parts into one string separated by ", ".

12. ptn
   The part number reference.
   Found in the RIGHT column, on a line containing "PTN:".
   Extract only the value after "PTN:".
   Note: OCR may misread certain characters (e.g. "S" as "$") — extract as-is.

13. reqna
    The requisition name.
    Found in the RIGHT column, on a line containing "REQNA:".
    Extract only the value after "REQNA:".

14. weight_value
   The numeric shipment weight (without unit).
   Found in the LEFT column, on a line containing "WEIGHT:" or "EIGHT:" (OCR sometimes drops the leading "W").
   Extract only the numeric part — e.g. "71.00".

15. weight_unit
   The unit of the shipment weight.
   Found on the same line as weight_value.
   Extract only the unit — e.g. "LBS" or "KG".

16. pieces
    The number of pieces.
    Found in the LEFT column, on a line containing "PIECES:".
    Extract only the numeric value after "PIECES:".

17. consignee
    The consignee name and full address.
    The label "CONSIGNEE:" appears in the CENTER column (OCR may read it as "CONSIGNEE!").
    The consignee content spans the lines BELOW that label in the CENTER column:
    - First line after the label: consignee company name
    - Following lines: street address, city/zip, country/state
    There may be blank lines or gaps between address parts — collect ALL CENTER column lines in the CONSIGNEE section until the billing table begins.
    Combine all parts into one string separated by ", ".

18. line_items
    All charge lines from the billing table.
    Found in the lower portion of the document after the header row containing "Service Description", "Amount", "Currency", "Exchange Rate", "Amount Inv Currency".
    For EACH data row in that table (stop before the TOTAL row), extract:
      - description: leftmost column (service name)
      - amount: second column (numeric value — digits and dots/commas only, no currency symbol)
      - currency: third column (3-letter code, e.g. "USD")
      - exchange_rate: fourth column (numeric value)
      - amount_inv_currency: fifth column (rightmost numeric value)
    Return as a JSON array of objects — one object per row.
    If no rows found, return [].
    Example:
    [{{"description": "Hand Carry", "amount": "4090.00", "currency": "USD", "exchange_rate": "1.00", "amount_inv_currency": "4090.00"}}]

19. invoice_total
    The total amount due.
    Found on the line containing "Total Amount Due".
    Extract only the numeric value — digits, dots and commas only, no currency symbol — e.g. "4,090.00".

20. invoice_currency
    The currency of the invoice total.
    Found on the same line as "Total Amount Due", or take it from the Currency column of the billing table.
    Extract only the 3-letter code — e.g. "USD".

21. tms_number
    Found on the line containing "TMS Number:".
    Extract the numeric value after "TMS Number:".

=== RULES ===
- If a field is missing, set its value to "N/A" (or [] for line_items).
- Do not infer or calculate any values; only extract what is present in the OCR.
- Do not provide any explanation, only the JSON.
- Return a single JSON array with one object.

[
  {{
    "supplier": ...,
    "invoice_no": ...,
    "invoice_date": ...,
    "bill_to": ...,
    "awb": ...,
    "pickup_date": ...,
    "shipper": ...,
    "ptn": ...,
    "reqna": ...,
    "weight_value": ...,
    "weight_unit": ...,
    "pieces": ...,
    "consignee": ...,
    "line_items": [...],
    "invoice_total": ...,
    "invoice_currency": ...,
    "tms_number": ...
  }}
]

Document text:
\"\"\"
{extracted_text}
\"\"\"
"""
        escaped_prompt = prompt.replace("'", "\\'")
        sql = f"""
            SELECT snowflake.cortex.complete(
                'llama3.1-70b',
                $$ {escaped_prompt} $$
            )
        """
        result = session.sql(sql).collect()
        summary_text = result[0][0]
        return summary_text.strip(), None
    except Exception as e:
        return None, f"❌ Failed to run Cortex Completion: {e}"


    Write data to excel

In [6]:
def insert_invoice_data(json_data, file_path, ocr_text, invoice_file):
    line_items = json_data.get("line_items", [])
    line_items_str = json.dumps(line_items, ensure_ascii=False) if line_items else "[]"

    row = {
        "CARRIER_NAME_RAW": json_data.get("supplier", ""),
        "INVOICE_ID": json_data.get("invoice_no", ""),
        "INVOICE_DATE": json_data.get("invoice_date", ""),
        "AWB": json_data.get("awb", ""),
        "PICKUP_DATE": json_data.get("pickup_date", ""),
        "SHIPPER": json_data.get("shipper", ""),
        "PTN": json_data.get("ptn", ""),
        "WEIGHT": json_data.get("weight", ""),
        "PIECES": json_data.get("pieces", ""),
        "CONSIGNEE": json_data.get("consignee", ""),
        "LINE_ITEMS": line_items_str,
        "INVOICE_TOTAL": json_data.get("invoice_total", ""),
        "TMS_NUMBER": json_data.get("tms_number", ""),
        "OCR_TEXT": ocr_text,
        "SOURCE_FILE": invoice_file,
    }

    df_new = pd.DataFrame([row])

    if os.path.exists(file_path):
        df_existing = pd.read_excel(file_path)
        df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    else:
        df_combined = df_new

    df_combined.to_excel(file_path, index=False)
    print(f"✅ Data written to {file_path}")


Image processing code to autorotate the images

In [7]:
def deskew_image_pil(pil_img, show=False):
    """
    Deskew image using horizontal projection profile.
    Scans angles from -15° to +15° and picks the one that maximises row-sum variance
    (well-aligned text produces sharp peaks in the row projection).
    """
    img = np.array(pil_img.convert("L"))

    # Binarise to isolate text pixels
    img_bin = cv2.adaptiveThreshold(
        img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8
    )

    h, w = img_bin.shape
    center = (w // 2, h // 2)

    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-15, 15.1, 0.5):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(
            img_bin, M, (w, h), flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT, borderValue=0,
        )
        score = float(np.var(np.sum(rotated, axis=1)))
        if score > best_score:
            best_score, best_angle = score, angle

    print(f"Detected skew angle: {best_angle:.1f}°")

    M = cv2.getRotationMatrix2D(center, best_angle, 1.0)
    rotated_img = cv2.warpAffine(
        img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE
    )
    pil_rotated = Image.fromarray(rotated_img)

    if show:
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(pil_img, cmap="gray" if pil_img.mode == "L" else None)
        axes[0].set_title("Before deskew")
        axes[0].axis("off")
        axes[1].imshow(pil_rotated, cmap="gray")
        axes[1].set_title(f"After deskew ({best_angle:.1f}°)")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

    return pil_rotated


In [8]:
def preprocess_image_pil(img):
    
    if img.mode != 'L':
        img = img.convert('L')
    
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(1.2)
        
    img_np = np.array(img)
    img_bin = cv2.adaptiveThreshold(
        img_np, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 21, 10
    )
    img = Image.fromarray(img_bin)

    return img

Image processing code to run tesseract OCR 

In [20]:
def _create_positioned_line(line_items, chars_per_pixel=12):
    """Reconstruct a single text line preserving original X-positions as spaces."""
    line_items.sort(key=lambda x: x["x"])
    if not line_items:
        return ""

    first = line_items[0]
    result = " " * max(0, first["x"] // chars_per_pixel) + first["text"]
    current_x = first["x"] + len(first["text"]) * chars_per_pixel

    for item in line_items[1:]:
        spaces = max(1, (item["x"] - current_x) // chars_per_pixel)
        result += " " * spaces + item["text"]
        current_x = item["x"] + len(item["text"]) * chars_per_pixel

    return result


def extract_with_real_positions(image_input, chars_per_pixel=12):
    """
    Run Tesseract OCR and reconstruct text with spatial layout preserved.
    Words are placed at their real pixel X positions so columns remain aligned.

    Args:
        image_input: file path (str) or PIL Image
        chars_per_pixel: pixel-to-character ratio controlling column spacing
    """
    image = Image.open(image_input) if isinstance(image_input, str) else image_input

    data = pytesseract.image_to_data(
        image,
        output_type=pytesseract.Output.DICT,
        config="--oem 1 --psm 11 --dpi 350",
    )

    # Keep all non-empty tokens (confidence filter disabled intentionally)
    tokens = [
        {"text": data["text"][i], "x": data["left"][i], "y": data["top"][i],
         "width": data["width"][i], "height": data["height"][i]}
        for i in range(len(data["text"]))
        if data["text"][i].strip()
    ]

    # Sort top-to-bottom, left-to-right and group into lines (±17 px tolerance)
    tokens.sort(key=lambda t: (t["y"], t["x"]))
    lines, current_line, current_y = [], [], None

    for token in tokens:
        if current_y is None or abs(token["y"] - current_y) < chars_per_pixel:
            current_line.append(token)
            current_y = token["y"]
        else:
            lines.append(_create_positioned_line(current_line, chars_per_pixel))
            current_line = [token]
            current_y = token["y"]

    if current_line:
        lines.append(_create_positioned_line(current_line, chars_per_pixel))

    return "\n".join(lines)


In [21]:
def _pdf_to_images(pdf_path, max_pages=None):
    """Convert PDF pages to PIL images using Poppler."""
    kwargs = {"dpi": 350, "poppler_path": POPPLER_PATH}
    if max_pages:
        kwargs.update({"first_page": 1, "last_page": max_pages})
    return convert_from_path(pdf_path, **kwargs)


def pdf_to_ocr_text(pdf_path, max_pages=None):
    """
    Convert all pages of a PDF to position-aware OCR text.
    Each page is wrapped with markers:
        === OCR: Full Page N (Real Positions) === ... === End of Page N ===
    """
    print(f"OCR: {os.path.basename(pdf_path)}"
          + (f" (first {max_pages} page(s))" if max_pages else ""))

    images = _pdf_to_images(pdf_path, max_pages)
    if not images:
        print("  ERROR: could not convert PDF to images.")
        return None

    pages = []
    for i, image in enumerate(images, 1):
        try:
            preprocessed = preprocess_image_pil(image)
        except Exception as exc:
            print(f"  WARN: preprocessing failed on page {i}: {exc} — using greyscale.")
            preprocessed = image.convert("L")

        page_text = extract_with_real_positions(preprocessed)
        pages.append(
            f"\n=== OCR: Full Page {i} (Real Positions) ===\n"
            f"{page_text.strip()}\n"
            f"=== End of Page {i} ==="
        )

    return "\n".join(pages).strip()


def get_numbered_ocr_text(pdf_path, max_pages=None):
    """Return OCR text with line numbers (useful for debugging LLM extraction)."""
    raw = pdf_to_ocr_text(pdf_path, max_pages)
    if not raw:
        return "ERROR: OCR failed."

    lines = raw.split("\n")
    numbered = [
        f"{i:3d}: {line}" if line.strip() else f"{i:3d}: [empty]"
        for i, line in enumerate(lines, 1)
    ]
    numbered.append(f"\nTotal lines: {len(lines)}")
    return "\n".join(numbered)


def batch_ocr_to_txt(input_dir, output_dir, max_pages=None):
    """Process all PDFs in input_dir, write one .txt OCR file per PDF to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"No PDF files found in: {input_dir}")
        return

    for filename in pdf_files:
        txt_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".txt")
        text = get_numbered_ocr_text(os.path.join(input_dir, filename), max_pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"  Saved: {txt_path}")


In [22]:
batch_ocr_to_txt(input_directory, output_directory, max_pages=1)

OCR: Courier Network Inc. Airfreight Invoice Data Point To Be Collected.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\output\Courier Network Inc. Airfreight Invoice Data Point To Be Collected.txt
OCR: US625053359.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\output\US625053359.txt
OCR: US626006966.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\output\US626006966.txt
OCR: US725037937.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\output\US725037937.txt
OCR: US725042730.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\Courier Network INC 1\output\US725042730.txt


In [15]:
import os

input_directory = r"C:\Users\jakub.klosowski\email\FAP\Courier Network INC 1"

def process_directory_ocr_stage1(input_dir, max_pages=1):
    """
    Stage 1 pipeline: for each PDF in input_dir
      1. Run OCR  →  position-aware text
      2. Send text to Cortex LLM  →  structured JSON
      3. Append result to Excel
      4. Collect all records for Snowflake staging insert
    """
    print(f"Scanning: {input_dir}")
    print("=" * 60)

    raw_invoices_data = []
    local_excel_path  = os.path.join(input_dir, "invoice_extract.xlsx")
    total_saved       = 0

    for filename in sorted(os.listdir(input_dir)):
        if not filename.lower().endswith(".pdf"):
            continue

        print(f"\nProcessing: {filename}")
        extracted_text = pdf_to_ocr_text(
            os.path.join(input_dir, filename), max_pages=max_pages
        )

        if not extracted_text:
            print(f"  ERROR: OCR returned nothing for {filename}")
            continue

        structured_output, error = ask_cortex_invoice_data(session, extracted_text)
        if error:
            print(f"  ERROR (Cortex): {error}")
            continue

        try:
            invoices = json.loads(structured_output)
        except json.JSONDecodeError as e:
            print(f"  ERROR (JSON parse): {e}\n  Raw response: {structured_output}")
            continue

        for invoice in invoices:
            invoice["filename"] = filename
            try:
                # insert_invoice_data(invoice, local_excel_path)
                total_saved += 1
            except Exception as exc:
                print(f"  ERROR (Excel): {exc}")
        print("✅ Structured Data extracted:")
        print(json.dumps(invoices, indent=2, ensure_ascii=False))

        raw_invoices_data.extend(invoices)

    print("=" * 60)
    print(f"Invoices extracted : {len(raw_invoices_data)}")
    print(f"Saved to Excel     : {total_saved}")
    return raw_invoices_data


raw_invoices_data = process_directory_ocr_stage1(input_directory, max_pages=1)


Scanning: C:\Users\jakub.klosowski\email\FAP\Courier Network INC 1

Processing: Courier Network Inc. Airfreight Invoice Data Point To Be Collected.pdf
OCR: Courier Network Inc. Airfreight Invoice Data Point To Be Collected.pdf (first 1 page(s))
✅ Structured Data extracted:
[
  {
    "supplier": "Courier Network Inc.",
    "invoice_no": "US725039316",
    "invoice_date": "30/11/25",
    "bill_to": "Autoliv ASP, Inc. 3350 Airport Road Ogden, UT 84405 USA",
    "awb": "11560543",
    "ship_date": "25/11/2025",
    "delivery_date": "26/11/25",
    "delivery_time": "14:55:00",
    "delivery_timezone": "N/A",
    "shipper": "NITTO, INC, 120 FORTUNE DR, FRANKFORT 40601, KY,US",
    "ptn": "892819",
    "reqna": "Hugo Acosta",
    "weight_value": "71.00",
    "weight_unit": "LBS",
    "pieces": "1",
    "consignee": "AUTOLIV STEERING WHEELS MEXICO S$ DE, CIRCUITO EL MARQUES NORTE #25, EL MARQUES QUERETARO 76246, QA,MX",
    "line_items": [
      {
        "description": "Hand Carry",
        "

In [ ]:
import uuid
import json


def _esc(val):
    if val is None or str(val).strip() in ('', 'N/A', 'n/a'):
        return 'NULL'
    return "'" + str(val).replace("'", "''") + "'"


def _parse_amount(s):
    try:
        return float(str(s).replace(',', ''))
    except Exception:
        return None


def _parse_weight_value(s):
    try:
        return float(str(s).strip().replace(',', '.'))
    except Exception:
        return None


def insert_invoices_to_staging(
    session,
    raw_invoices_data,
    header_table='STG_INVOICE_HEADER',
    charge_table='STG_INVOICE_CHARGE_LINES',
    ref_table='STG_INVOICE_REFERENCES',
):
    if not raw_invoices_data:
        print("No data to insert.")
        return

    existing = {
        r[0] for r in session.sql(f"SELECT INVOICE_ID FROM {header_table}").collect() if r[0]
    }
    print(f"Already in {header_table}: {len(existing)} invoice(s)")

    inserted_h = inserted_c = inserted_r = 0

    for inv in raw_invoices_data:
        invoice_id = str(inv.get('invoice_no') or '').strip()
        if not invoice_id or invoice_id == 'N/A':
            print(f"  SKIP (no invoice_no): {inv.get('filename', '?')}")
            continue
        if invoice_id in existing:
            print(f"  SKIP (duplicate): {invoice_id}")
            continue

        weight_val  = _parse_weight_value(inv.get('weight_value'))
        weight_unit = inv.get('weight_unit')
        amount_val  = _parse_amount(inv.get('invoice_total'))
        currency    = inv.get('invoice_currency')

        line_items = inv.get('line_items') or []
        if isinstance(line_items, str):
            try:
                line_items = json.loads(line_items)
            except Exception:
                line_items = []

        ocr_json    = json.dumps(inv, ensure_ascii=False)
        ocr_escaped = "'" + ocr_json.replace("'", "''") + "'"

        # ── STG_INVOICE_HEADER ─────────────────────────────────────────────
        session.sql(f"""
            INSERT INTO {header_table} (
                INVOICE_ID, INVOICE_DATE, DUE_DATE,
                SHIP_DATE, DELIVERY_DATE, DELIVERY_TIME, DELIVERY_TIMEZONE,
                CARRIER_NAME_RAW, BILL_TO_RAW,
                INVOICE_TOTAL, CURRENCY,
                TRANSPORT_MODE,
                SHIP_FROM, SHIP_TO,
                WEIGHT_VALUE, WEIGHT_UNIT,
                SOURCE_CARRIER, OCR_RAW_PAYLOAD,
                LOAD_TIMESTAMP, PROCESSING_STATUS
            )
            SELECT
                {_esc(invoice_id)},
                TRY_TO_DATE({_esc(inv.get('invoice_date'))}, 'DD/MM/YYYY'),
                NULL,
                TRY_TO_DATE({_esc(inv.get('ship_date'))}, 'DD/MM/YYYY'),
                TRY_TO_DATE({_esc(inv.get('delivery_date'))}, 'DD/MM/YYYY'),
                {_esc(inv.get('delivery_time'))},
                NULL,
                {_esc(inv.get('supplier'))},
                {_esc(inv.get('bill_to'))},
                {amount_val if amount_val is not None else 'NULL'},
                {_esc(currency)},
                'Air',
                {_esc(inv.get('shipper'))},
                {_esc(inv.get('consignee'))},
                {weight_val if weight_val is not None else 'NULL'},
                {_esc(weight_unit)},
                {_esc(inv.get('supplier'))},
                PARSE_JSON({ocr_escaped}),
                CURRENT_TIMESTAMP(),
                'NEW'
        """).collect()
        inserted_h += 1

        # ── STG_INVOICE_CHARGE_LINES ──────────────────────────────────────
        for seq, item in enumerate(line_items, 1):
            if not isinstance(item, dict):
                continue
            charge_amount = _parse_amount(item.get('amount'))
            session.sql(f"""
                INSERT INTO {charge_table} (
                    CHARGE_LINE_ID, INVOICE_ID, SOURCE_CARRIER,
                    CHARGE_SEQUENCE, CHARGE_DESCRIPTION_RAW,
                    CHARGE_QUANTITY, CHARGE_UNIT_PRICE,
                    CHARGE_AMOUNT, CURRENCY
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('supplier'))},
                    {seq},
                    {_esc(item.get('description'))},
                    NULL,
                    NULL,
                    {charge_amount if charge_amount is not None else 'NULL'},
                    {_esc(item.get('currency'))}
                )
            """).collect()
            inserted_c += 1

        # ── STG_INVOICE_REFERENCES ────────────────────────────────────────
        references = [
            ('AWB',        inv.get('awb')),
            ('PTN',        inv.get('ptn')),
            ('TMS_NUMBER', inv.get('tms_number')),
            ('AUTOLIV PERSON CONCERNED',      inv.get('reqna')),
        ]
        for ref_type, ref_val in references:
            session.sql(f"""
                INSERT INTO {ref_table} (
                    REFERENCE_ID, INVOICE_ID, SOURCE_CARRIER,
                    REFERENCE_TYPE, REFERENCE_VALUE
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get('supplier'))},
                    {_esc(ref_type)},
                    {_esc(ref_val)}
                )
            """).collect()
            inserted_r += 1

        existing.add(invoice_id)
        print(f"  OK  {invoice_id}")

    print(f"\nSTG_INVOICE_HEADER:       {inserted_h} row(s)")
    print(f"STG_INVOICE_CHARGE_LINES: {inserted_c} row(s)")
    print(f"STG_INVOICE_REFERENCES:   {inserted_r} row(s)")


# ── Execute ───────────────────────────────────────────────────────────────────
session.sql("USE DATABASE DEV_SCM_DB").collect()
session.sql("USE SCHEMA OUT_TMS").collect()

insert_invoices_to_staging(session, raw_invoices_data)


Already in STG_INVOICE_HEADER: 6 invoice(s)
  OK  US725039316
  OK  US625053359
  OK  US626006966
  OK  US725037937
  OK  US725042730

STG_INVOICE_HEADER:       5 row(s)
STG_INVOICE_CHARGE_LINES: 7 row(s)
STG_INVOICE_REFERENCES:   20 row(s)
